# 05 · Real Transactions - Kaggle Groceries Dataset
**Owner: P3 (AI)** · Smart Shelf Market Basket Analysis

Validates the AI half on **real purchase data**: 38,765 purchase rows → ~15k baskets
([Heeral Dedhia, Kaggle](https://www.kaggle.com/datasets/heeraldedhia/groceries-dataset)).

1. Apriori → strongest and most *surprising* ("beers & diapers"-style) rules
2. KNN next-item Hit@3 vs popularity on real baskets (Experiment 5b)

In [ ]:
# ============================================================
# SETUP - identical in every notebook. Works on Colab, Mac, Windows.
# ============================================================
import os, sys, subprocess
from pathlib import Path

# Colab only: where to get the code from.
#   - GitHub (recommended): set with  python tools/set_repo_url.py <url>
#   - "" : the product-identification-correlation folder is in Google Drive at MyDrive/product-identification-correlation
REPO_URL = "https://github.com/behlah-rgb/product-identification-correlation.git"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if REPO_URL:
        repo = Path("/content/product-identification-correlation")
        url = REPO_URL
        try:  # private repo: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    else:
        repo = Path("/content/drive/MyDrive/product-identification-correlation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo))
os.chdir(repo)

from src import env
env.summary()

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import mining
from src.env import DATASETS_DIR, RESULTS_DIR

csv = DATASETS_DIR / "groceries" / "Groceries_dataset.csv"
if not csv.exists():
    import kagglehub, shutil
    path = kagglehub.dataset_download("heeraldedhia/groceries-dataset")
    csv.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(next(__import__("pathlib").Path(path).glob("*.csv")), csv)
    # If this fails: download the CSV from the Kaggle page and put it at the path above.

raw = pd.read_csv(csv)
print(raw.shape); raw.head()

## 1. Build baskets (one customer on one day = one basket)

In [ ]:
raw["itemDescription"] = raw["itemDescription"].str.strip().str.lower()
baskets = raw.groupby(["Member_number", "Date"])["itemDescription"].apply(lambda s: sorted(set(s))).tolist()
matrix = mining.to_matrix(baskets)
print(f"{len(baskets)} baskets | {matrix.shape[1]} unique items | mean size {np.mean([len(b) for b in baskets]):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.Series([len(b) for b in baskets]).value_counts().sort_index().plot.bar(ax=ax[0], rot=0); ax[0].set_title("Basket size")
matrix.mean().sort_values().tail(15).plot.barh(ax=ax[1]); ax[1].set_title("Top 15 items by support")
plt.tight_layout(); plt.show()

## 2. Apriori

In [ ]:
MIN_SUPPORT, MIN_LIFT = 0.001, 1.0
itemsets, rules = mining.mine_rules(matrix, MIN_SUPPORT, MIN_LIFT, max_len=3)
print(f"{len(itemsets)} frequent itemsets | {len(rules)} rules with lift >= {MIN_LIFT}")
top = mining.readable(rules, 15)
top.to_csv(RESULTS_DIR / "groceries_top_rules.csv", index=False)
top

In [ ]:
# "Impossible correlations": strong lift between items that are individually NOT that common,
# with enough support to not be pure chance (>= ~15 baskets).
min_count = 15 / len(matrix)
surprising = rules[(rules.support >= min_count) & (rules.lift >= 1.5)].copy()
surprising = surprising[surprising["antecedents"].apply(len) + surprising["consequents"].apply(len) == 2]
print(f"{len(surprising)} pair rules with lift >= 1.5 and support >= {min_count:.4f}")
mining.readable(surprising.sort_values("lift", ascending=False), 15)

In [ ]:
# Lift heatmap for the 15 most popular items
top_items = matrix.mean().sort_values(ascending=False).head(15).index
sup = matrix[top_items].mean().values
joint = (matrix[top_items].T.astype(int) @ matrix[top_items].astype(int)).values / len(matrix)
lift = joint / np.outer(sup, sup); np.fill_diagonal(lift, np.nan)
plt.figure(figsize=(8, 7)); plt.imshow(lift, cmap="coolwarm", vmin=0.5, vmax=1.5)
plt.xticks(range(15), top_items, rotation=90); plt.yticks(range(15), top_items); plt.colorbar(label="lift")
plt.title("Pairwise lift - 15 most popular items"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "groceries_lift_heatmap.png", dpi=150); plt.show()

## 3. Experiment 5b - KNN next-item on real baskets

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(matrix)); cut = int(0.8 * len(matrix))
train_m, test_m = matrix.iloc[idx[:cut]], matrix.iloc[idx[cut:]]

rows = [{"model": "Popularity baseline", "k": "-", "Hit@3": mining.popularity_hit_rate(train_m, test_m)}]
for metric in ["hamming", "jaccard"]:
    for k in [5, 20, 50]:
        rows.append({"model": f"KNN ({metric})", "k": k, "Hit@3": mining.knn_hit_rate(train_m, test_m, k, metric)})
exp5b = pd.DataFrame(rows).round(3)
exp5b.to_csv(RESULTS_DIR / "exp5b_knn_groceries.csv", index=False)
exp5b

**For the paper (P3):** real-data rules table, the "surprising pairs" table (the beers-and-diapers discussion),
lift heatmap, and Experiment 5b. Real grocery baskets are small (~2.6 items), which limits KNN - an honest
finding worth discussing next to the simulated results.